# 25 GSE65858_enhanced_KM_curves_ROUND2

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.plotting import add_at_risk_counts
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("VERIFIED_CORRECT_scores.csv")

expected = {
    "Score_lambda_min": (1.81, 0.016), "Score_lambda_1se": (2.65, 0.024),
    "Score_Toustrup": (1.27, 0.179), "Score_Eustace": (0.99, 0.94),
    "Score_Lendahl": (1.42, 0.172), "Score_RSI": (3.18, 0.245),
}
all_ok = True
for col, (exp_hr, exp_p) in expected.items():
    d = df[["OS_months","Event",col]].dropna()
    m = CoxPHFitter(); m.fit(d, duration_col="OS_months", event_col="Event")
    s = m.summary.loc[col]
    hr, p = np.exp(s["coef"]), s["p"]
    ok = abs(hr-exp_hr) < 0.02 and abs(p-exp_p) < 0.005
    all_ok = all_ok and ok
    print(f"{col}: HR={hr:.3f} (expected {exp_hr}), p={p:.3f} (expected {exp_p}) -- {'OK' if ok else 'MISMATCH'}")

assert all_ok, "Score verification failed -- do not proceed with stale/incorrect data"
print("\nAll six signatures verified. Proceeding to generate figures.")

In [ ]:
df["OS_months_5yr"] = np.minimum(df["OS_months"], 60.0)
df["Event_5yr"] = np.where(df["OS_months"] <= 60.0, df["Event"], 0)

signatures = [
    ("Score_lambda_min", "Our lambda.min (PRIMARY, 32 genes)", "KM_5yr_lambda_min_ENHANCED.png"),
    ("Score_lambda_1se", "Our lambda.1se (SENSITIVITY, 13 genes)", "KM_5yr_lambda_1se_ENHANCED.png"),
    ("Score_Toustrup", "Toustrup (15 genes)", "KM_5yr_Toustrup_ENHANCED.png"),
    ("Score_Eustace", "Eustace/NIMRAD (adapted, 25 of 26 genes)", "KM_5yr_Eustace_ENHANCED.png"),
    ("Score_Lendahl", "Lendahl (adapted, 28 of 30 genes)", "KM_5yr_Lendahl_ENHANCED.png"),
    ("Score_RSI", "RSI (10 genes, within-patient rank, exploratory)", "KM_5yr_RSI_ENHANCED.png"),
]

for col, label, fname in signatures:
    d = df[["OS_months_5yr", "Event_5yr", col]].dropna()
    med = d[col].median()
    high, low = d[d[col] >= med], d[d[col] < med]

    fig, ax = plt.subplots(figsize=(7.5, 6.2))
    kmf_h = KaplanMeierFitter()
    kmf_h.fit(high["OS_months_5yr"], high["Event_5yr"], label=f"High (n={len(high)})")
    kmf_h.plot_survival_function(ax=ax, ci_show=True, show_censors=True, censor_styles={"marker": "|", "ms": 8})

    kmf_l = KaplanMeierFitter()
    kmf_l.fit(low["OS_months_5yr"], low["Event_5yr"], label=f"Low (n={len(low)})")
    kmf_l.plot_survival_function(ax=ax, ci_show=True, show_censors=True, censor_styles={"marker": "|", "ms": 8})

    ax.set_xlabel("Overall Survival (Months)"); ax.set_ylabel("Survival Probability")
    ax.set_title(f"GSE65858 (5-yr censored): {label}", fontsize=10)
    ax.set_xlim(0, 60); ax.grid(alpha=0.3)
    add_at_risk_counts(kmf_h, kmf_l, ax=ax, rows_to_show=["At risk"])
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {fname}")

print("\nAll 6 figures generated from verified-correct scores.")